# Two-stage MTech specialist detector (GPU) — the recipe behind 0.818

Trains **MTech-only** (1,756 imgs) with the paper's two-stage protocol, run to
full convergence on GPU (the original was CPU-limited):
- **Stage 1:** YOLOv8n from COCO, 100 ep @ 640, safe augmentation.
- **Stage 2:** fine-tune @ 768 (busbar recall-boost), lower LR.
- **Eval with test-time augmentation (TTA)** for a free robustness gain.

Goal: match/beat the 0.818 baseline by doing the *specialist* right (not adding
diverse data, which we proved lowers the MTech number).

Upload `ev_mtech_data.zip` (in ~/Downloads) to Drive anywhere; the notebook finds it.

In [ ]:
import torch, subprocess
print('CUDA:', torch.cuda.is_available(), '|', subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'))
!pip install -q ultralytics

In [ ]:
# Mount Drive + auto-find the zip
from google.colab import drive
import glob, subprocess
drive.mount('/content/drive')
hits = glob.glob('/content/drive/MyDrive/**/ev_mtech_data.zip', recursive=True)
assert hits, 'ev_mtech_data.zip not found in Drive — upload it first.'
print('Found:', hits[0])
subprocess.run(['unzip','-q','-o',hits[0],'-d','/content/'])
!echo 'train:' $(ls /content/ev_mtech/images/train/*.jpg | wc -l) ' test:' $(ls /content/ev_mtech/images/test/*.jpg | wc -l)

In [ ]:
# STAGE 1 — 100 epochs @ 640 (cold start from COCO, paper-safe augmentation)
from ultralytics import YOLO
YAML = '/content/ev_mtech/dataset.yaml'
s1 = YOLO('yolov8n.pt')
s1.train(data=YAML, epochs=100, imgsz=640, batch=32, device=0,
         optimizer='SGD', lr0=0.01, weight_decay=0.0005,
         hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5,
         degrees=2.0, translate=0.06, fliplr=0.5, patience=30,
         project='/content/runs', name='stage1', exist_ok=True)

In [ ]:
# STAGE 2 — fine-tune the Stage-1 best @ 768, lower LR (busbar recall-boost)
s2 = YOLO('/content/runs/stage1/weights/best.pt')
s2.train(data=YAML, epochs=40, imgsz=768, batch=16, device=0,
         optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005,
         hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5,
         degrees=2.0, translate=0.06, fliplr=0.5, patience=15,
         project='/content/runs', name='stage2', exist_ok=True)

In [ ]:
# EVALUATE on the MTech test set — plain and with TTA (augment=True)
print('=== Stage 2, plain ===')
m = s2.val(data=YAML, split='test', imgsz=768, device=0, name='s2_test', project='/content/runs', exist_ok=True)
print(f'  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')
for i,c in m.names.items(): print(f'    {c}: mAP50={m.box.ap50[i]:.3f}')
print('=== Stage 2 + TTA ===')
mt = s2.val(data=YAML, split='test', imgsz=768, device=0, augment=True, name='s2_test_tta', project='/content/runs', exist_ok=True)
print(f'  mAP50={mt.box.map50:.3f}  mAP50-95={mt.box.map:.3f}   (baseline 0.818 / 0.557)')

In [ ]:
# Save + download the Stage-2 weights
from google.colab import files
best='/content/runs/stage2/weights/best.pt'
!cp "$best" /content/drive/MyDrive/ev_detector_twostage_best.pt
print('Saved to Drive: MyDrive/ev_detector_twostage_best.pt'); files.download(best)

## Read the result
- If Stage-2 (esp. +TTA) mAP50 reaches or beats **0.818**, the specialist-done-right
  worked — that's your stronger detector.
- Send me the four numbers (plain + TTA). If it beats baseline, drop `best.pt` at
  `models/detector/stage2_recall_boost/weights/best.pt` and I'll verify locally.